## Fine-tuning a model with the Trainer API

In [2]:
!pip install -U datasets transformers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
# get the dataset
from datasets import load_dataset
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [2]:
#  basic version check

import datasets
import huggingface_hub
import transformers

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("transformers:", transformers.__version__)

datasets: 5.0.1
huggingface_hub: 1.30.0
transformers: 5.16.1


In [9]:
# choose a pre-trained model
checkpoint = "bert-base-uncased"


In [11]:
# get the tokenizer
# tokenizer = converts text into something the model can understand
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print(tokenizer)

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})


In [12]:
# tokenize the dataset
def tokenize_function(example):
  return tokenizer(
      example["sentence1"],
      example["sentence2"],
      truncation=True
  )


In [13]:
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched = True
)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Original dataset
      ->
"I love dogs"
"The dog is cute"
      ->
Tokenizer
      ->
Numbers/tokens
      ->
Tokenized dataset

In [14]:
# create the model
# now we need a model that can do classification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)
model

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Why num_labels=2?

Because our task has two possible answers:

0 → Not similar
1 → Similar

this model needs to choose between 2 classes

        BERT
         ↓
    New classifier
         ↓
     0 or 1

In [15]:
# TrainingArguments
# Now we tell the trainer how we want training to happen
# "test-trainer" is basically the directory where training outputs/checkpoints can be saved.
from transformers import TrainingArguments

training_args = TrainingArguments(
    "test-trainer",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
)

| Setting            | Meaning                           |
| ------------------ | --------------------------------- |
| `learning_rate`    | How big each learning step is     |
| `batch_size`       | How many examples at once         |
| `num_train_epochs` | How many times to see the dataset |
| `eval_strategy`    | When to evaluate                  |
| `fp16`             | Use mixed precision               |


In [19]:
# create a trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],

)

Model          → What should I train?

Training args  → How should I train it?

Train dataset  → What should I learn from?

Validation     → How should I test myself?

Tokenizer      → How should I process the text?

In [ ]:
# train model
trainer.train()

# but how do we known if the model is good?

This is where evaluation comes in.

Suppose the model predicts:

Actual:     1
Prediction: 1

Good! ✅

But:

Actual:     1
Prediction: 0

Wrong. ❌

We need metrics to measure how well it performs.

In [28]:
! pip install evaluate
import numpy as np
import evaluate

metric = evaluate.load("glue", "mrpc")

def compute_metrics(eval_preds):
    logits, labels = eval_preds

    predictions = np.argmax(logits, axis=-1)

    return metric.compute(
        predictions=predictions,
        references=labels
    )

In [29]:
eval_strategy="epoch"

In [30]:
training_args = TrainingArguments(
    "test-trainer",
    eval_strategy="epoch"
)

Train for 1 epoch
      
      ↓

Evaluate
      
      ↓
Train next epoch
      
      ↓
Evaluate
     
      ↓
...

It uses eval_strategy="epoch" or "steps" to control when evaluation occurs

Data → Tokenize → Model → Trainer → Train → Evaluate